# 1. Data Processing

## Importing all the nesessary Libraries

In [8]:
import pandas as pd
from pathlib import Path

# Path to the repo's dataset folder
path = Path("Datasets")
# Combines your folder name and the *.csv pattern into a list of CSV files.
all_files = sorted(path.glob("*.csv"))

#The loop (processing each file)
## li-- Initializes an empty list to hold the data from each file temporarily.
li =[]

for filename in all_files:
    df = pd.read_csv(filename, sep=";", index_col = None, header=0)
    #Tagging the Data 
    df['Patient_ID'] = filename.stem
    li.append(df)

#Merging everything

combined_activity = pd.concat(li, axis= 0,ignore_index = True)
combined_activity.to_csv("Combined_Datasets.csv", index=False)

In [9]:
# checking the Shape and top rows
print(f"Total rows and columns:{combined_activity.shape}")
print(combined_activity.head())

Total rows and columns:(309392, 9)
                  time  glucose  calories  heart_rate  steps  basal_rate  \
0  2018-06-13T18:40:00    332.0    6.3595   82.322835   34.0    0.091667   
1  2018-06-13T18:45:00    326.0    7.7280   83.740157    0.0    0.091667   
2  2018-06-13T18:50:00    330.0    4.7495   80.525180    0.0    0.091667   
3  2018-06-13T18:55:00    324.0    6.3595   89.129032   20.0    0.091667   
4  2018-06-13T19:00:00    306.0    5.1520   92.495652    0.0    0.075000   

   bolus_volume_delivered  carb_input Patient_ID  
0                     0.0         0.0  HUPA0001P  
1                     0.0         0.0  HUPA0001P  
2                     0.0         0.0  HUPA0001P  
3                     0.0         0.0  HUPA0001P  
4                     0.0         0.0  HUPA0001P  


In [10]:
combined_activity.head

<bound method NDFrame.head of                        time     glucose  calories  heart_rate  steps  \
0       2018-06-13T18:40:00  332.000000   6.35950   82.322835   34.0   
1       2018-06-13T18:45:00  326.000000   7.72800   83.740157    0.0   
2       2018-06-13T18:50:00  330.000000   4.74950   80.525180    0.0   
3       2018-06-13T18:55:00  324.000000   6.35950   89.129032   20.0   
4       2018-06-13T19:00:00  306.000000   5.15200   92.495652    0.0   
...                     ...         ...       ...         ...    ...   
309387  2022-05-18T11:55:00  109.333333  10.79280  104.171171    0.0   
309388  2022-05-18T12:00:00  114.000000   9.80346  103.442623    0.0   
309389  2022-05-18T12:05:00  118.666667   5.66622   95.542857    0.0   
309390  2022-05-18T12:10:00  123.333333   5.57628   91.381356    0.0   
309391  2022-05-18T12:15:00  128.000000   5.57628   99.257812    0.0   

        basal_rate  bolus_volume_delivered  carb_input Patient_ID  
0         0.091667                   

In [13]:
# Create a 5-minute interval dataset using time as the reference
combined_activity['time'] = pd.to_datetime(combined_activity['time'], errors='coerce')
combined_activity = combined_activity.dropna(subset=['time'])

aggregation_rules = {
    'glucose': 'mean',
    'calories': 'sum',
    'heart_rate': 'mean',
    'steps': 'sum',
    'basal_rate': 'mean',
    'bolus_volume_delivered': 'sum',
    'carb_input': 'sum',
}

combined_activity_5min = (
    combined_activity.sort_values(['Patient_ID', 'time'])
    .set_index('time')
    .groupby('Patient_ID')
    .resample('15min')
    .agg(aggregation_rules)
    .reset_index()
)

combined_activity_5min.to_csv('Combined_Datasets_15min_avg.csv', index=False)

print(f'Total rows and columns: {combined_activity_5min.shape}')
print('Saved file: Combined_Datasets_5min_avg.csv')
combined_activity_5min.head()


Total rows and columns: (103147, 9)
Saved file: Combined_Datasets_5min_avg.csv


,Patient_ID,time,glucose,calories,heart_rate,steps,basal_rate,bolus_volume_delivered,carb_input
0,HUPA0001P,2018-06-13 18:30:00,332.000000,6.3595,82.322835,34.0,0.091667,0.0,0.0
1,HUPA0001P,2018-06-13 18:45:00,326.666667,18.8370,84.464790,20.0,0.091667,0.0,0.0
2,HUPA0001P,2018-06-13 19:00:00,310.333333,21.8960,91.945092,54.0,0.075000,0.0,0.0
3,HUPA0001P,2018-06-13 19:15:00,296.333333,20.1250,89.675048,77.0,0.075000,0.0,0.0
4,HUPA0001P,2018-06-13 19:30:00,265.333333,37.2715,100.805189,156.0,0.075000,0.0,0.0
